In [28]:
import pandas as pd
import numpy as np
from scipy import stats
import seaborn as sns
import matplotlib.pyplot as plt
import re

In [29]:
DATA_ORIGINAL_CSV = '../data/preprocessed_shorts.csv'          
DATA_VISUAL_CSV = '../data/features_visuais.csv'     
DATA_MERGED_CSV = '../data/dataset_multimodal.csv'

In [30]:
def load_and_merge():
    print("Carregando as bases de dados...")
    df_original = pd.read_csv(DATA_ORIGINAL_CSV)
    df_visual = pd.read_csv(DATA_VISUAL_CSV)
    
    df_merged = pd.merge(df_original, df_visual, on='videoId', how='inner')
    
    colunas_numericas = ['viewCount', 'likeCount', 'commentCount', 
                         'colorfulness', 'brightness', 'text_ratio', 'face_ratio']
    for col in colunas_numericas:
        df_merged[col] = pd.to_numeric(df_merged[col], errors='coerce')
        
    df_merged = df_merged.dropna(subset=['viewCount', 'colorfulness', 'isShorts'])
    
    df_merged['Formato'] = np.where(df_merged['isShorts'] == True, 'Shorts', 'Video Longo')
    
    return df_merged

In [31]:
def analise_formatos_mann_whitney(df):
    print("\nDIFERENÇAS DE FORMATO: SHORTS vs VÍDEOS LONGOS")
    df_shorts = df[df['Formato'] == 'Shorts']
    df_longos = df[df['Formato'] == 'Video Longo']
    
    features = ['colorfulness', 'text_ratio', 'face_ratio', 'brightness']
    
    for feature in features:
        stat, p_value = stats.mannwhitneyu(df_shorts[feature].dropna(), df_longos[feature].dropna(), alternative='two-sided')
        print(f"\nFeature: {feature}")
        print(f"  Média Shorts: {df_shorts[feature].mean():.4f} | Média Longos: {df_longos[feature].mean():.4f}")
        print(f"  P-Valor: {p_value:.4e} " + ("(Significativa)" if p_value < 0.05 else "(Não significativa)"))

In [32]:
def analise_correlacao_engajamento(df):
    print("\nCORRELAÇÃO DE ENGAJAMENTO (Spearman)")
    formatos = ['Shorts', 'Video Longo']
    features_visuais = ['colorfulness', 'text_ratio', 'face_ratio']
    
    for formato in formatos:
        print(f"\n[{formato}] Correlação com ViewCount:")
        df_recorte = df[df['Formato'] == formato]
        
        for feature in features_visuais:
            corr, p_value = stats.spearmanr(df_recorte[feature], df_recorte['viewCount'], nan_policy='omit')
            print(f"  {feature}: r = {corr:.4f} (p = {p_value:.4e})")

In [33]:
def analise_cross_modal_topicos(df):
    print("\nCROSS-MODAL: TÓPICOS vs. THUMBNAILS")
    if 'algorithmTopics' in df.columns:
        top_topicos = df['algorithmTopics'].value_counts().head(5).index
        df_top = df[df['algorithmTopics'].isin(top_topicos)]
        
        resumo = df_top.groupby('algorithmTopics')[['colorfulness', 'text_ratio']].mean().reset_index()
        print("\nMédias visuais por principais tópicos do algoritmo:")
        print(resumo.to_string(index=False))
    else:
        print("Coluna 'algorithmTopics' não encontrada no DataFrame.")

In [34]:
def gerar_visualizacoes(df):
    print("\nGERANDO GRÁFICOS")
    sns.set_theme(style="whitegrid")
    
    # Gráfico 1: Saturação por Formato
    plt.figure(figsize=(8, 6))
    sns.boxplot(x='Formato', y='colorfulness', data=df, palette='Set2')
    plt.title('Distribuição de Saturação (Colorfulness) por Formato de Vídeo')
    plt.ylabel('Índice de Saturação')
    plt.xlabel('Formato')
    plt.savefig('boxplot_colorfulness_formatos.png')
    plt.close()
    
    # Gráfico 2: Quantidade de Texto por Formato
    plt.figure(figsize=(8, 6))
    sns.barplot(x='Formato', y='text_ratio', data=df, estimator=np.mean, errorbar=('ci', 95), palette='pastel')
    plt.title('Proporção Média de Texto na Capa por Formato')
    plt.ylabel('Taxa de Ocupação de Texto')
    plt.xlabel('Formato')
    plt.savefig('barplot_textratio_formatos.png')
    plt.close()
    
    print("Gráficos salvos: 'boxplot_colorfulness_formatos.png' e 'barplot_textratio_formatos.png'")

In [35]:
def analise_indice_clickbait(df):
    print("\nO ÍNDICE DE CLICKBAIT (Taxas de Conversão vs. Estética)")
    
    df_click = df.copy()
    df_click['taxa_curtidas'] = df_click['likeCount'] / df_click['viewCount']
    df_click['taxa_comentarios'] = df_click['commentCount'] / df_click['viewCount']
    
    df_click = df_click.replace([np.inf, -np.inf], np.nan).dropna(subset=['taxa_curtidas', 'taxa_comentarios'])
    
    features_visuais = ['colorfulness', 'text_ratio', 'brightness']
    taxas = ['taxa_curtidas', 'taxa_comentarios']
    
    for taxa in taxas:
        print(f"\nCorrelação de Spearman com {taxa.upper()}:")
        for feature in features_visuais:
            corr, p_value = stats.spearmanr(df_click[feature], df_click[taxa])
            sig = "Significativa" if p_value < 0.05 else "Não sig."
            print(f"  {feature}: r = {corr:.4f} (p = {p_value:.4e}) [{sig}]")

In [ ]:
def analise_variacao_categoria(df):
    print("\nVARIAÇÃO ESTÉTICA POR CATEGORIA")
    
    if 'categoryId' not in df.columns:
        print("Coluna 'categoryId' não encontrada.")
        return

    # Calcula as médias por categoria
    resumo_cat = df.groupby('categoryId')[['colorfulness', 'brightness', 'text_ratio']].mean()
    resumo_cat['quantidade_videos'] = df['categoryId'].value_counts()
    
    # Filtra categorias com número muito baixo de vídeos (ex: < 10) para não sujar a análise
    resumo_cat = resumo_cat[resumo_cat['quantidade_videos'] >= 10].sort_values(by='colorfulness', ascending=False)
    
    print("\nMédias visuais pelas principais categorias (ordenado por Saturação):")
    print(resumo_cat.round(4).to_string())
    
    # Teste de Kruskal-Wallis
    grupos_categorias = [grupo['colorfulness'].values for name, grupo in df.groupby('categoryId') if len(grupo) >= 10]
    if len(grupos_categorias) > 1:
        stat, p_value = stats.kruskal(*grupos_categorias)
        print(f"\nTeste Kruskal-Wallis para Colorfulness entre categorias: p-valor = {p_value:.4e}")

In [ ]:
def analise_sinergia_texto(df):
    print("\nSINERGIA DE TEXTO (Título vs. Thumbnail)")
    
    df_txt = df.dropna(subset=['title', 'text_ratio']).copy()
    df_txt['tamanho_titulo'] = df_txt['title'].str.len()
    
    corr, p_value = stats.spearmanr(df_txt['tamanho_titulo'], df_txt['text_ratio'])
    sig = "Significativa" if p_value < 0.05 else "Não sig."
    
    print(f"Correlação de Spearman (Tamanho do Título x Taxa de Texto na Capa):")
    print(f"  r = {corr:.4f} (p = {p_value:.4e}) [{sig}]")

In [38]:
def parse_youtube_duration(duration_str):
    """ Função auxiliar para converter o formato PT1H2M10S do YouTube para segundos inteiros """
    if pd.isna(duration_str):
        return np.nan
    hours = re.search(r'(\d+)H', duration_str)
    minutes = re.search(r'(\d+)M', duration_str)
    seconds = re.search(r'(\d+)S', duration_str)
    
    h = int(hours.group(1)) if hours else 0
    m = int(minutes.group(1)) if minutes else 0
    s = int(seconds.group(1)) if seconds else 0
    return h * 3600 + m * 60 + s

In [39]:
def analise_estetica_tempo(df):
    print("\nA ESTÉTICA DO TEMPO (Duração vs. Identidade Visual)")
    
    # Foca apenas em vídeos longos (Shorts têm restrição rígida de tempo, o que enviesa essa análise)
    df_longos = df[df['Formato'] == 'Video Longo'].copy()
    
    if 'duration' not in df_longos.columns:
        print("Coluna 'duration' não encontrada.")
        return
        
    # Converte duração para segundos numéricos
    df_longos['duration_seconds'] = df_longos['duration'].apply(parse_youtube_duration)
    df_longos = df_longos.dropna(subset=['duration_seconds'])
    
    labels_quartis = ['1. Muito Curto', '2. Curto', '3. Médio', '4. Longo']
    df_longos['quartil_tempo'] = pd.qcut(df_longos['duration_seconds'], q=4, labels=labels_quartis)
    
    resumo_tempo = df_longos.groupby('quartil_tempo')[['colorfulness', 'text_ratio', 'brightness', 'duration_seconds']].mean()
    
    print("\nMédias visuais por faixas de duração de vídeo:")
    print(resumo_tempo.round(4).to_string())

In [40]:
df_final = load_and_merge()
    
df_final = load_and_merge()
analise_formatos_mann_whitney(df_final)
analise_correlacao_engajamento(df_final)
#analise_cross_modal_topicos(df_final)

analise_indice_clickbait(df_final)
analise_variacao_categoria(df_final)
analise_sinergia_texto(df_final)
analise_estetica_tempo(df_final)

gerar_visualizacoes(df_final)

Carregando as bases de dados...
Carregando as bases de dados...

DIFERENÇAS DE FORMATO: SHORTS vs VÍDEOS LONGOS

Feature: colorfulness
  Média Shorts: 31.2426 | Média Longos: 63.8293
  P-Valor: 0.0000e+00 (Significativa)

Feature: text_ratio
  Média Shorts: 0.0446 | Média Longos: 0.1560
  P-Valor: 0.0000e+00 (Significativa)

Feature: face_ratio
  Média Shorts: 0.0000 | Média Longos: 0.0000
  P-Valor: 1.0000e+00 (Não significativa)

Feature: brightness
  Média Shorts: 60.9101 | Média Longos: 125.4643
  P-Valor: 0.0000e+00 (Significativa)

CORRELAÇÃO DE ENGAJAMENTO (Spearman)

[Shorts] Correlação com ViewCount:
  colorfulness: r = 0.1043 (p = 4.7476e-15)
  text_ratio: r = 0.0256 (p = 5.5450e-02)
  face_ratio: r = nan (p = nan)

[Video Longo] Correlação com ViewCount:
  colorfulness: r = 0.1806 (p = 2.2698e-25)
  text_ratio: r = 0.1880 (p = 2.1194e-27)
  face_ratio: r = nan (p = nan)

O ÍNDICE DE CLICKBAIT (Taxas de Conversão vs. Estética)

Correlação de Spearman com TAXA_CURTIDAS:
  colo

C:\Users\Nathan\AppData\Local\Temp\ipykernel_12368\1240032425.py:11: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corr, p_value = stats.spearmanr(df_recorte[feature], df_recorte['viewCount'], nan_policy='omit')
C:\Users\Nathan\AppData\Local\Temp\ipykernel_12368\4166864897.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  resumo_tempo = df_longos.groupby('quartil_tempo')[['colorfulness', 'text_ratio', 'brightness', 'duration_seconds']].mean()
C:\Users\Nathan\AppData\Local\Temp\ipykernel_12368\1955836137.py:7: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x='Formato', y='colorfulness', data=df, pal

Gráficos salvos: 'boxplot_colorfulness_formatos.png' e 'barplot_textratio_formatos.png'


C:\Users\Nathan\AppData\Local\Temp\ipykernel_12368\1955836137.py:16: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x='Formato', y='text_ratio', data=df, estimator=np.mean, errorbar=('ci', 95), palette='pastel')
